In [33]:
import math, requests, urllib3, os
import pandas as pd
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from datetime import datetime

# ── BYPASS SSL VERIFICATION (TEMPORARY USE ONLY) ──────────────────────────────
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
REQUESTS_KW = {"verify": False, "timeout": 15}

# ─── Elo knobs ──────────────────────────────────────────────────────────────
BASE_RATING, HOME_FIELD_ELO = 1500.0, 55.0

# ─── Betting knobs ───────────────────────────────────────────────────────────
SPREAD_SIGMA = 13.45           # stdev of NFL scoring margin (points)
SPREAD_EDGE_THRESHOLD = 0.5    # min edge to recommend a spread bet
TOTAL_EDGE_THRESHOLD = 1.5     # min edge (pts) to recommend Over/Under

# ─── Historical blending knobs (NEW) ────────────────────────────────────────
USE_PREV_SEASON = True         # set False to ignore prior season entirely
PREV_SEASON_WEIGHT = 0.35      # weight on previous season when blending PPG/OPPG

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 24)

# ---------- Normal CDF / inverse ----------
def phi(x):  # Φ(x)
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def phi_inv(p):  # Φ^-1(p) — Acklam approx (no SciPy)
    if p <= 0.0 or p >= 1.0:
        raise ValueError("phi_inv expects 0 < p < 1")
    a = [-3.969683028665376e+01,  2.209460984245205e+02, -2.759285104469687e+02,
          1.383577518672690e+02, -3.066479806614716e+01,  2.506628277459239e+00]
    b = [-5.447609879822406e+01,  1.615858368580409e+02, -1.556989798598866e+02,
          6.680131188771972e+01, -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e+00,
         -2.549732539343734e+00,  4.374664141464968e+00,  2.938163982698783e+00]
    d = [ 7.784695709041462e-03,  3.224671290700398e-01,  2.445134137142996e+00,
          3.754408661907416e+00]
    plow, phigh = 0.02425, 1 - 0.02425
    if p < plow:
        q = math.sqrt(-2.0 * math.log(p))
        return (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
               ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1.0)
    if p > phigh:
        q = math.sqrt(-2.0 * math.log(1.0 - p))
        return -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1.0)
    q = p - 0.5
    r = q*q
    return (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
           (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1.0)

# ---------- Market↔Model transforms ----------
def model_fair_home_spread_from_prob(p_home, sigma=SPREAD_SIGMA):
    """Convert model P(home win) to fair spread for home (negative => home favorite)."""
    z = phi_inv(float(p_home))
    mu = sigma * z
    return -mu

def format_market_line(home_team, away_team, home_spread):
    """Favorite with minus; if away fav, show away -pts; 0 => Pick'em; None => No Line."""
    if home_spread is None:
        return "No Line"
    if home_spread == 0:
        return "Pick'em"
    if home_spread < 0:
        return f"{home_team} {home_spread:.1f}"  # e.g., PHI -7.5
    return f"{away_team} {-home_spread:.1f}"     # e.g., KC -3.5 (away fav)

# ---------- Data fetch: scoreboard ----------
def fetch_espn_scoreboard():
    url = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
    r = requests.get(url, **REQUESTS_KW); r.raise_for_status()
    data = r.json()
    # ESPN sometimes omits season/week if out of window; be resilient:
    season_year = (data.get("season", {}) or {}).get("year") or datetime.now().year
    week_num = (data.get("week", {}) or {}).get("number")
    rows = []
    for ev in data.get("events", []):
        comp = ev.get("competitions", [{}])[0]
        comps = comp.get("competitors", [])
        if not comps: 
            continue
        home = next((c for c in comps if c.get("homeAway") == "home"), None)
        away = next((c for c in comps if c.get("homeAway") == "away"), None)
        if home is None or away is None:
            continue
        odds = (comp.get("odds") or [{}])[0]
        def to_float(x):
            try:
                return float(x) if x not in (None, "", "PK") else None
            except Exception:
                return None
        rows.append({
            "season": season_year,
            "week": week_num,
            "home_team": home.get("team", {}).get("abbreviation"),
            "away_team": away.get("team", {}).get("abbreviation"),
            "home_spread": to_float(odds.get("spread")),    # negative => home favored
            "total_ou": to_float(odds.get("overUnder")),
        })
    return pd.DataFrame(rows)

# ---------- Data fetch: team IDs ----------
def fetch_team_ids():
    """Map ESPN abbreviation → team id."""
    url = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/teams"
    r = requests.get(url, **REQUESTS_KW); r.raise_for_status()
    data = r.json()
    id_map = {}
    sports = data.get("sports") or []
    leagues = (sports[0].get("leagues") if sports else []) or []
    teams = (leagues[0].get("teams") if leagues else []) or []
    for item in teams:
        team = item.get("team", {})
        abbr = team.get("abbreviation")
        tid  = team.get("id")
        if abbr and tid:
            id_map[abbr] = tid
    return id_map

# ---------- Current-season PPG/OPPG via team endpoint (best-effort) ----------
def _extract_ppg_from_stats_payload(payload: dict):
    """
    Try to locate 'Points Per Game' and 'Opponent Points Per Game' in ESPN's team payload.
    The structure can vary; we search generically by displayName.
    """
    ppg = oppg = None

    def walk(node):
        nonlocal ppg, oppg
        if isinstance(node, dict):
            dn = node.get("displayName") or node.get("shortDisplayName")
            name = (node.get("name") or "").lower()
            if dn == "Points Per Game" or name == "pointspergame":
                v = node.get("value") or node.get("displayValue")
                try: ppg = float(v)
                except: pass
            if dn in ("Opponent Points Per Game","Opp. Points Per Game") or name in ("opponentpointspergame","opppointspergame","oppPointsPerGame".lower()):
                v = node.get("value") or node.get("displayValue")
                try: oppg = float(v)
                except: pass
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for it in node:
                walk(it)

    walk(payload)
    return ppg, oppg

def fetch_team_ppg_oppg(team_id: str):
    """
    Query ESPN team endpoint for CURRENT stats and return (PPG, OPPG).
    We try multiple 'enable' flags to maximize compatibility.
    """
    for enable in ("team.stats", "statistics", ""):
        url = f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/teams/{team_id}"
        if enable:
            url += f"?enable={enable}"
        r = requests.get(url, **REQUESTS_KW); r.raise_for_status()
        data = r.json()
        ppg, oppg = _extract_ppg_from_stats_payload(data)
        if ppg is not None and oppg is not None:
            return ppg, oppg
    return None, None  # fallback

def build_ppg_table(abbrs):
    """Return dict: abbr -> {'PPG': x, 'OPPG': y} for CURRENT season, best-effort; missing -> None."""
    id_map = fetch_team_ids()
    out = {}
    for ab in set(abbrs):
        tid = id_map.get(ab)
        if not tid:
            out[ab] = {"PPG": None, "OPPG": None}
            continue
        ppg, oppg = fetch_team_ppg_oppg(tid)
        out[ab] = {"PPG": ppg, "OPPG": oppg}
    return out

# ---------- PREVIOUS-season PPG/OPPG via schedule (NEW) ----------------------
def fetch_team_ppg_oppg_prev_by_schedule(team_id: str, season: int):
    """
    Compute prior-season PPG/OPPG from the team schedule (more reliable than stats flags).
    Includes regular + postseason if present. Skips games without scores.
    """
    url = f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/teams/{team_id}/schedule?season={season}"
    r = requests.get(url, **REQUESTS_KW); r.raise_for_status()
    data = r.json()

    games = 0
    pts_for = 0.0
    pts_against = 0.0

    for ev in data.get("events", []):
        comp = (ev.get("competitions") or [{}])[0]
        competitors = comp.get("competitors") or []
        if len(competitors) < 2:
            continue

        # Find our team and opponent in this game by team id
        me = None
        opp = None
        for c in competitors:
            team = c.get("team", {}) or {}
            if str(team.get("id")) == str(team_id):
                me = c
            else:
                opp = c if opp is None else opp

        if me is None or opp is None:
            continue

        # Scores are strings; skip if missing
        s_me = me.get("score")
        s_opp = opp.get("score")
        try:
            s_me_f = float(s_me)
            s_opp_f = float(s_opp)
        except (TypeError, ValueError):
            continue

        games += 1
        pts_for += s_me_f
        pts_against += s_opp_f

    if games == 0:
        return None, None
    return pts_for / games, pts_against / games

def build_prev_ppg_table(abbrs, season_prev: int):
    """Return dict: abbr -> {'PPG': x, 'OPPG': y} for PREVIOUS season (from schedule)."""
    id_map = fetch_team_ids()
    out = {}
    for ab in set(abbrs):
        tid = id_map.get(ab)
        if not tid:
            out[ab] = {"PPG": None, "OPPG": None}
            continue
        ppg, oppg = fetch_team_ppg_oppg_prev_by_schedule(tid, season_prev)
        out[ab] = {"PPG": ppg, "OPPG": oppg}
    return out

def blend_ppg_tables(curr_tbl: dict, prev_tbl: dict, w_prev: float):
    """
    Blend current and previous season PPG/OPPG: blended = (1-w_prev)*current + w_prev*previous.
    If one side is missing, fall back to the other; if both missing => None.
    """
    out = {}
    keys = set(curr_tbl.keys()) | set(prev_tbl.keys())
    for k in keys:
        c = curr_tbl.get(k, {})
        p = prev_tbl.get(k, {})
        c_ppg, c_oppg = c.get("PPG"), c.get("OPPG")
        p_ppg, p_oppg = p.get("PPG"), p.get("OPPG")

        def mix(a, b):
            if a is None and b is None:
                return None
            if a is None:
                return b
            if b is None:
                return a
            return (1.0 - w_prev) * a + w_prev * b

        out[k] = {"PPG": mix(c_ppg, p_ppg), "OPPG": mix(c_oppg, p_oppg)}
    return out

# ---------- Simple Elo-ish snapshot to create variation ----------
def snapshot_features(df):
    ratings = defaultdict(lambda: BASE_RATING)
    out = []
    for _, row in df.iterrows():
        h, a = row["home_team"], row["away_team"]
        h_elo, a_elo = ratings[h], ratings[a]
        out.append({
            "home_team": h,
            "away_team": a,
            "elo_diff_pre": (h_elo + HOME_FIELD_ELO) - a_elo,  # just to create variation
            "home_spread": row["home_spread"],
            "total_ou": row["total_ou"],
        })
    return pd.DataFrame(out)

# ---------- Deterministic picks ----------
def pick_spread(home_team, away_team, market_home_spread, model_home_prob,
                sigma=SPREAD_SIGMA, threshold_pts=SPREAD_EDGE_THRESHOLD):
    """Return a valid ATS pick string (team with proper +/- line) or 'Pass'."""
    if market_home_spread is None or model_home_prob is None:
        return "Pass"
    fair_home = model_fair_home_spread_from_prob(model_home_prob, sigma)
    edge_pts = fair_home - market_home_spread
    if edge_pts >= threshold_pts:
        # Like the dog: take points with AWAY at away's line
        away_line = -market_home_spread
        return f"Bet {away_team} {'+' if away_line>0 else ''}{away_line:.1f}"
    if edge_pts <= -threshold_pts:
        # Like the favorite: lay/take with HOME at home's line
        home_line = market_home_spread
        return f"Bet {home_team} {'+' if home_line>0 else ''}{home_line:.1f}"
    return "Pass"

def pick_total(home_abbr, away_abbr, market_ou, ppg_table, threshold_pts=TOTAL_EDGE_THRESHOLD):
    """
    Model total = avg(offense vs opp defense).
    Returns 'Bet Over', 'Bet Under', or 'Pass'.
    Uses blended PPG/OPPG if provided.
    """
    if market_ou is None:
        return "No Line"

    h = ppg_table.get(home_abbr, {})
    a = ppg_table.get(away_abbr, {})
    h_ppg, h_oppg = h.get("PPG"), h.get("OPPG")
    a_ppg, a_oppg = a.get("PPG"), a.get("OPPG")

    # If any missing, we can't make a modelled call
    if None in (h_ppg, h_oppg, a_ppg, a_oppg):
        return "Pass"

    model_total = 0.5 * (h_ppg + a_oppg) + 0.5 * (a_ppg + h_oppg)

    if model_total >= market_ou + threshold_pts:
        return "Bet Over"
    if model_total <= market_ou - threshold_pts:
        return "Bet Under"
    return "Pass"

# ---------- Main ----------
if __name__ == "__main__":
    sb = fetch_espn_scoreboard()
    if sb.empty:
        print("No games found."); raise SystemExit(0)

    # Figure seasons (current from scoreboard; previous = current-1)
    season_current = int(sb["season"].dropna().iloc[0]) if "season" in sb.columns and sb["season"].notna().any() else datetime.now().year
    season_prev = season_current - 1

    feats = snapshot_features(sb)

    # Build CURRENT PPG/OPPG table (best-effort via team endpoint)
    team_abbrs = list(set(feats["home_team"]).union(set(feats["away_team"])))
    ppg_curr = build_ppg_table(team_abbrs)

    # Build PREVIOUS PPG/OPPG table (from previous season schedule) (NEW)
    if USE_PREV_SEASON:
        prev_tbl = build_prev_ppg_table(team_abbrs, season_prev)
        # Blend current + previous (NEW)
        ppg_table = blend_ppg_tables(ppg_curr, prev_tbl, PREV_SEASON_WEIGHT)
    else:
        ppg_table = ppg_curr

    # Add a prior-season net-strength feature to spread model (NEW)
    # net = PPG - OPPG (prev season); diff = home_net - away_net
    def prior_net_diff(h_abbr, a_abbr):
        if not USE_PREV_SEASON:
            return 0.0
        h = (prev_tbl or {}).get(h_abbr, {})
        a = (prev_tbl or {}).get(a_abbr, {})
        h_ppg, h_oppg = h.get("PPG"), h.get("OPPG")
        a_ppg, a_oppg = a.get("PPG"), a.get("OPPG")
        if None in (h_ppg, h_oppg, a_ppg, a_oppg):
            return 0.0
        return (h_ppg - h_oppg) - (a_ppg - a_oppg)

    feats["prior_net_ppg_diff"] = [
        prior_net_diff(h, a) for h, a in zip(feats["home_team"], feats["away_team"])
    ]

    # Tiny model: use elo_diff_pre and prior_net_ppg_diff to create home-win prob variation (UPDATED)
    X = feats[["elo_diff_pre", "prior_net_ppg_diff"]].values
    y = [1 if s is not None and s < 0 else 0 for s in feats["home_spread"]]  # 1=home favored by book
    if len(set(y)) > 1:
        try:
            lr = LogisticRegression()
            p_home = lr.fit(X, y).predict_proba(X)[:, 1]
        except Exception:
            p_home = [0.5] * len(X)
    else:
        p_home = [0.5] * len(X)

    # Final, clean view
    df = feats.copy()
    df["Matchup"] = df["away_team"] + " @ " + df["home_team"]
    df["Market Line"] = [
        format_market_line(h, a, s)
        for h, a, s in zip(df["home_team"], df["away_team"], df["home_spread"])
    ]
    df["Total (O/U)"] = df["total_ou"]

    # Picks
    df["Spread Pick"] = [
        pick_spread(h, a, s, pm)
        for h, a, s, pm in zip(df["home_team"], df["away_team"], df["home_spread"], p_home)
    ]
    df["Totals Pick"] = [
        pick_total(h, a, ou, ppg_table)
        for h, a, ou in zip(df["home_team"], df["away_team"], df["Total (O/U)"])
    ]

    # Select/format columns
    out = df[["Matchup","Market Line","Total (O/U)","Spread Pick","Totals Pick"]].copy()
    out["Total (O/U)"] = pd.to_numeric(out["Total (O/U)"], errors="coerce").round(1)

    # Sort: actionable bets (either column not 'Pass') on top
    out["__has_pick__"] = ((out["Spread Pick"] != "Pass") | (out["Totals Pick"] != "Pass")).astype(int)
    out = out.sort_values(["__has_pick__", "Matchup"], ascending=[False, True]).drop(columns="__has_pick__").reset_index(drop=True)

    # Display & save
    try:
        from IPython.display import display as _display
        _display(out)
    except Exception:
        print(out.to_string(index=False))


,Matchup,Market Line,Total (O/U),Spread Pick,Totals Pick
0,ARI @ NO,ARI -6.5,43.5,Bet NO +6.5,Pass
1,BAL @ BUF,BUF -1.5,50.5,Bet BUF -1.5,Pass
2,CAR @ JAX,JAX -3.5,46.5,Bet CAR +3.5,Pass
3,CIN @ CLE,CIN -5.5,47.5,Bet CLE +5.5,Pass
4,DAL @ PHI,PHI -8.5,47.5,Bet DAL +8.5,Pass
5,HOU @ LAR,LAR -3.5,43.5,Bet HOU +3.5,Pass
6,KC @ LAC,KC -3.5,46.5,Bet LAC +3.5,Pass
7,MIA @ IND,IND -1.5,46.5,Bet IND -1.5,Pass
8,MIN @ CHI,MIN -1.5,43.5,Bet CHI +1.5,Pass
9,NYG @ WSH,WSH -6.5,45.5,Bet NYG +6.5,Pass


In [35]:
blend_ppg_tables

<function __main__.blend_ppg_tables(curr_tbl: dict, prev_tbl: dict, w_prev: float)>